___
<a href='https://honglab.ai'><p style="text-align:center;"><img src='https://lh3.googleusercontent.com/lY3ySXooSmwsq5r-mRi7uiypbo0Vez6pmNoQxMFhl9fmZJkRHu5lO2vo7se_0YOzgmDyJif9fi4_z0o3ZFdwd8NVSWG6Ea80uWaf3pOHpR4GHGDV7kaFeuHR3yAjIJjDgfXMxsvw=w2400'  class="center" width="50%" height="50%"/></p></a>
___
<center><em>Content Copyright by HongLab, Inc.</em></center>

# 엔그램으로 문장 만들기

**해당 강의:** 파트1 챕터2 「04-06. 문장 생성」 · 「04-07. 형태소 분석」

SF 소설 27편으로 문장을 만듭니다. 글자 단위로 먼저 만들고, 형태소 단위로 바꾼 다음, 원문을 베끼지 않도록 백오프를 넣습니다.

- 실행 시간: 30초 안팎 (CPU)
- 메모리: 약 1GB
- 설치할 것: 형태소 분석기 키위(Kiwi) — `pip install kiwipiepy`

## 데이터 읽기와 문장 자르기

`data/sf_novels.txt`는 위키문헌에서 받은 SF 소설 27편에서 출판 정보·한자·특수 기호를 지우고 한 파일로 합친 것입니다.

강의의 C 코드와 같은 규칙으로 문장을 자릅니다.
- 한글, 공백, 쉼표, 문장부호(`.` `!` `?`)만 남깁니다.
- 줄바꿈은 공백으로 바꿉니다.
- 빈 줄(문단 경계)과 문장부호에서 문장을 자릅니다.
- 한글이 한 글자도 없는 문장은 버립니다.

In [1]:
import re

text = open("data/sf_novels.txt", encoding="utf-8").read()
not_allowed = re.compile(r"[^가-힣 ,.!?]")

sentences = []
for para in re.split(r"\n\s*\n", text):              # 빈 줄 = 문단 경계
    para = not_allowed.sub("", " ".join(para.split()))  # 줄바꿈은 공백으로 바꾸고 허용한 글자만 남김
    para = re.sub(r" +", " ", para)
    for s in re.findall(r"[^.!?]+[.!?]*", para):       # 문장부호에서 자르기
        s = s.strip()
        if re.search(r"[가-힣]", s):
            sentences.append(s)

sentence_set = set(sentences)       # 만든 문장이 원문에 그대로 있는지 확인할 때 씁니다
print(f"문장 {len(sentences):,}개 · 글자 {sum(map(len, sentences)):,}개 · 글자 종류 {len(set(''.join(sentences))):,}개\n")
for s in sentences[1:4]:
    print(s)

문장 64,297개 · 글자 1,829,987개 · 글자 종류 1,502개

랄프는 우주모를 쓰고, 리자놀의 우주선으로 뛰어들어가 조종실 문을 열었습니다.
거기에는 미터가 넘는 리자놀의 큰 몸집이 그대로 바닥에 넘어져 있는 것이 보였습니다.
리자놀은 죽어 있었습니다.


## 글자 단위로 문장 만들기

단어를 만들 때와 코드가 같습니다. 달라진 점은 두 가지입니다.
- 공백, 쉼표, 마침표도 한 글자로 셉니다. 그래야 띄어쓰기와 문장부호가 있는 문장이 나옵니다.
- `<BOS>`와 `<EOS>`가 단어가 아니라 문장의 시작과 끝을 표시합니다.

`generate`는 문장의 시작 부분(`prompt`)도 받을 수 있게 만들었습니다. 이 함수를 뒤의 형태소 단위에서도 그대로 씁니다.

In [2]:
import random
from collections import Counter, defaultdict

BOS, EOS = "<BOS>", "<EOS>"

def count_ngrams(seqs, n):
    table = defaultdict(Counter)         # 문맥(앞 토큰 n-1개) → Counter(다음 토큰: 횟수)
    for seq in seqs:
        toks = [BOS] * (n - 1) + list(seq) + [EOS]
        for i in range(len(toks) - n + 1):
            table[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return table

def generate(table, n, rng, prompt=(), max_len=200):
    out = list(prompt)
    context = [BOS] * (n - 1) + out
    while len(out) < max_len:
        counter = table.get(tuple(context[-(n - 1):]))
        if not counter:                  # 표에 없는 문맥이면 멈춥니다
            break
        nxt = rng.choices(list(counter), weights=list(counter.values()))[0]
        if nxt == EOS:
            break
        out.append(nxt)
        context.append(nxt)
    return out

n=3, 5, 7로 200문장씩 만들어 앞의 다섯 개를 보고, 원문과 똑같은 문장이 몇 개인지도 셉니다.

In [3]:
for n in (3, 5, 7):
    table = count_ngrams(sentences, n)
    rng = random.Random(0)
    made = ["".join(generate(table, n, rng)) for _ in range(200)]
    copied = sum(s in sentence_set for s in made)
    print(f"--- {n}-그램 · 원문과 똑같은 문장: 200개 중 {copied}개")
    for s in made[:5]:
        print("  ", s)
    print()
    del table

--- 3-그램 · 원문과 똑같은 문장: 200개 중 10개
   임시 상당히 다른 순간 헐의 뜻도 굉장히 대로 위험을 구름이었다고 있어.
   텔레비전을 맛있다고 싶어하기 스리다는 털이 또 하였다.
   하고 싶다.
   출발해 보라색 셔터를 발견하셨기 때에는 집에서 그를 아닙니다.
   짐은 무대로 의하라는 맑은 공룡인 채 서로 가지 않았니?



--- 5-그램 · 원문과 똑같은 문장: 200개 중 12개
   임시로 끼워 줄게.
   당신이 가지를 줍는 것을 알 수가 없다.
   다시 한번 지나가다가 문득 소리를 내저으며 엄숙한 가운데 풀 같은 것을 불쾌하기 짝이 없는 것 같다.
   도로시, 내 손을 힘주어 잡고 정원을 산보하고 있을 텐데....
   그러나 웃음소리가 나더니 갑자기 새가 죽었다오.



--- 7-그램 · 원문과 똑같은 문장: 200개 중 33개
   임시로 끼워 맞춘 우주선이니까 더욱 국장님은 당신의 자유지요.
   바위 터의 언덕은 전송하는 사람들이 애독하고 프린스턴 대학에 입학하여 변호사가 되었습니다.
   용감하게 나서서 라비노 정신 병원으로 가서 사방을 찾고 있는 중이었습니다.
   그러니 도는 동안에 기계가 고장이 났지만, 상아 막대기가 몇 개 실려 있었다.
   아니, 또 한 대의 승용차와 같은 것을 집어들고는 깜짝 놀라며 황급히 다음 얘기를 삼켜 버리고 말았다.



3-그램은 얼핏 문장처럼 보이지만 뜻이 통하지 않습니다. 5-그램부터 읽히는 문장이 나오고, 7-그램은 원문의 문체가 느껴집니다.
대신 n이 커질수록 원문을 그대로 베낀 문장도 늘어납니다.

## 형태소 단위로 바꾸기

한국어는 조사와 어미가 붙는 교착어입니다. 공백으로 자른 어절을 단위로 쓰면 「학교가」, 「학교를」, 「학교는」이 전부 다른 단어가 되어 어휘가 폭발적으로 늘어납니다.

In [4]:
eojeol_vocab = Counter(w.strip(",.!?") for s in sentences for w in s.split())
print(f"어절 종류: {len(eojeol_vocab):,}개")
print("「학교」로 시작하는 어절:", [w for w in eojeol_vocab if w.startswith("학교")])

어절 종류: 76,700개
「학교」로 시작하는 어절: ['학교', '학교를', '학교의', '학교에', '학교에서', '학교는', '학교가', '학교에서는', '학교에서도', '학교에도', '학교놀이를', '학교로', '학교와', '학교도']


형태소는 뜻을 가진 가장 작은 말의 단위입니다. 형태소 분석기 키위(Kiwi)로 문장을 형태소로 나누면 형태소마다 품사 태그가 붙습니다(`NNG` 일반 명사, `JKS` 주격 조사, `JKO` 목적격 조사, `EF` 종결 어미 등).

In [5]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
for s in ["반갑습니다", "학교가 학교를 학교는"]:
    print(s, "→", [(t.form, t.tag) for t in kiwi.tokenize(s)])

반갑습니다 → [('반갑', 'VA-I'), ('습니다', 'EF')]
학교가 학교를 학교는 → [('학교', 'NNG'), ('가', 'JKS'), ('학교', 'NNG'), ('를', 'JKO'), ('학교', 'NNG'), ('는', 'JX')]


모든 문장을 형태소로 나누고, 형태소마다 숫자 ID를 붙여서 문장을 ID 목록으로 저장합니다. 현대 LLM의 토크나이저도 토큰을 숫자 ID로 바꿔서 다룹니다.

In [6]:
vocab, itos = {}, []                     # (형태소, 품사) → ID,  ID → (형태소, 품사)

def token_id(tok):
    if tok not in vocab:
        vocab[tok] = len(itos)
        itos.append(tok)
    return vocab[tok]

token_sents = [[token_id((t.form, t.tag)) for t in result] for result in kiwi.tokenize(sentences)]

print(f"토큰 {sum(map(len, token_sents)):,}개 · 형태소 종류 {len(itos):,}개 (어절 종류는 {len(eojeol_vocab):,}개)\n")
print("원문:", sentences[1])
print("ID  :", token_sents[1][:12], "...")
print("토큰:", [itos[i] for i in token_sents[1][:12]], "...")

토큰 1,046,776개 · 형태소 종류 16,295개 (어절 종류는 76,700개)

원문: 랄프는 우주모를 쓰고, 리자놀의 우주선으로 뛰어들어가 조종실 문을 열었습니다.
ID  : [21, 22, 23, 24, 10, 25, 26, 5, 27, 1, 13, 28] ...
토큰: [('랄프', 'NNP'), ('는', 'JX'), ('우주', 'NNG'), ('모', 'NNG'), ('를', 'JKO'), ('쓰', 'VV'), ('고', 'EC'), (',', 'SP'), ('리자놀', 'NNP'), ('의', 'JKG'), ('우주선', 'NNG'), ('으로', 'JKB')] ...


만든 ID 목록을 문장으로 되돌릴 때는 `kiwi.join`을 씁니다. 형태소를 이어 붙이고 띄어쓰기도 알아서 넣어 줍니다.
다만 복원한 띄어쓰기가 원문과 조금 다를 때가 있습니다. 그래서 만든 문장이 원문과 똑같은지 확인할 때는 문자열이 아니라 토큰 ID 목록끼리 비교합니다.

In [7]:
def to_text(ids):
    return kiwi.join([itos[i] for i in ids])

token_sent_set = set(map(tuple, token_sents))    # 만든 문장이 원문과 똑같은지 확인할 때 씁니다

print("원문:", sentences[1])
print("복원:", to_text(token_sents[1]))

원문: 랄프는 우주모를 쓰고, 리자놀의 우주선으로 뛰어들어가 조종실 문을 열었습니다.
복원: 랄프는 우주 모를 쓰고, 리자놀의 우주선으로 뛰어들어가 조종실 문을 열었습니다.


## 형태소 단위로 문장 만들기

`count_ngrams`와 `generate`를 그대로 씁니다. 글자 대신 형태소 ID가 들어갈 뿐입니다.
2-그램부터 5-그램까지의 표를 한꺼번에 만들어 둡니다. 뒤의 백오프에서 모두 씁니다.

In [8]:
tables = {n: count_ngrams(token_sents, n) for n in range(2, 6)}

rng = random.Random(0)
for n in (3, 5):
    made = [generate(tables[n], n, rng, max_len=100) for _ in range(200)]
    copied = sum(tuple(ids) in token_sent_set for ids in made)
    print(f"--- 형태소 {n}-그램 · 원문과 똑같은 문장: 200개 중 {copied}개")
    for ids in made[:5]:
        print("  ", to_text(ids) + ("  [원문]" if tuple(ids) in token_sent_set else ""))
    print()

--- 형태소 3-그램 · 원문과 똑같은 문장: 200개 중 9개
   통신을 읽고 있는 전자 로봇 헬렌과 러셀의 제이 스코아에 나와서 깃대 위에 있던 괴물의 무기로도 연기가 아주 조심스럽게 부탁해 온 소년 지구인은 당신 편이다.
   에 어록까지 마중 나왔다.
   그럴 필요가 있다!
   가령 일거리를 이것저것 궁리하고 있던 사나이가 나타나고 있으면 싸울 태세를 갖춘 내가 내려온 여신이라 생각하느냐고 우리들의 별들을 선택한다.
   전신이 쓰리고 아픔을 받아야한답니다.

--- 형태소 5-그램 · 원문과 똑같은 문장: 200개 중 71개
   세 사람은 함께 전자 계량 실을 나와 이스트 농장을 지나 출입구로 향했다.  [원문]
   맛있었다.  [원문]
   새로운 스카이라크 호에 사용될 아레낙 금속이란 무색 투명한 것이었다.
   울지 마.  [원문]
   그놈들은 으르렁거리는 것이 아닐까?



## 시작 단어 고정하기: 「돈을」

시작 부분을 「돈을」로 고정하고 5-그램으로 30문장을 만듭니다. 서로 다른 문장과 원문과 똑같은 문장이 각각 몇 개인지 셉니다.

In [9]:
prompt = [vocab[(t.form, t.tag)] for t in kiwi.tokenize("돈을")]
print("시작 토큰:", [itos[i] for i in prompt], "\n")

def report(made):
    copied = sum(tuple(ids) in token_sent_set for ids in made)
    print(f"{len(made)}문장 중 서로 다른 문장 {len(set(map(tuple, made)))}개 · 원문과 똑같은 문장 {copied}개\n")
    for ids in made[:10]:
        print("  ", to_text(ids) + ("  [원문]" if tuple(ids) in token_sent_set else ""))

rng = random.Random(1)
made = [generate(tables[5], 5, rng, prompt, max_len=100) for _ in range(30)]
report(made)

시작 토큰: [('돈', 'NNG'), ('을', 'JKO')] 

30문장 중 서로 다른 문장 25개 · 원문과 똑같은 문장 5개

   돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하는 거예요?
   돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하는 우리들의 기분이 젊은 추장의 표정으로 봐서 말입니다.
   돈을 많이 가지고 있다는 걸 알게 되면, 모두 그와 같은 고성능 폭약이 일시에 폭발한 데 대해 과학자도 고개를 갸웃거리고 있다.
   돈을 많이 가지고 있는 열쇠로 쇠 문을 열었다.
   돈을 두 배로 걸지 않겠어?  [원문]
   돈을 많이 가지고 있는 꿈인데, 찰리에게 있어서는 이 대지에서 외부로 통하는 길은 나타나지 않았다.
   돈을 주지 않느니라.
   돈을 많이 가지고 있다는 것도.
   돈을 주지 않았으므로 옥상에서 빗물을 모아 조금씩 사용했다.
   돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어한 진짜 목적, 곧 남아메리카 여행의 진상을, 특히 알겠소?


원문에 없는 문장도 사실은 원문의 조각을 이어 붙인 것입니다. 원문에 없는 문장마다 앞부분이 가장 길게 겹치는 원문 문장을 찾아봅니다.

In [10]:
import os

# 원문도 kiwi.join으로 되돌려야 띄어쓰기까지 같은 조건으로 비교할 수 있습니다
originals = [to_text(t) for t in token_sents if t[:len(prompt)] == prompt]
print(f"원문에서 「돈을」로 시작하는 문장: {len(originals)}개\n")

shown = 0
for ids in dict.fromkeys(map(tuple, made)):     # 중복을 빼고 순서대로
    if ids in token_sent_set:
        continue
    s = to_text(ids)
    best = max(originals, key=lambda o: len(os.path.commonprefix([s, o])))
    k = len(os.path.commonprefix([s, best]))
    print("만든 문장  :", s)
    print("닮은 원문  :", best)
    print(f"→ 앞의 「{s[:k]}」까지 원문과 같고 그 뒤를 다르게 이었습니다\n")
    shown += 1
    if shown == 3:
        break

원문에서 「돈을」로 시작하는 문장: 7개

만든 문장  : 돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하는 거예요?
닮은 원문  : 돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하거나 하는 사람은 한 사람도 없는 것 같았다.
→ 앞의 「돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하」까지 원문과 같고 그 뒤를 다르게 이었습니다

만든 문장  : 돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하는 우리들의 기분이 젊은 추장의 표정으로 봐서 말입니다.
닮은 원문  : 돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하거나 하는 사람은 한 사람도 없는 것 같았다.
→ 앞의 「돈을 모은다거나, 즐겁게 지내는 시간을 가지고 싶어하」까지 원문과 같고 그 뒤를 다르게 이었습니다

만든 문장  : 돈을 많이 가지고 있다는 걸 알게 되면, 모두 그와 같은 고성능 폭약이 일시에 폭발한 데 대해 과학자도 고개를 갸웃거리고 있다.
닮은 원문  : 돈을 많이 가지고 있는 부인이 있기에 나는 그것이 부러워서 바라보고 있었어요.
→ 앞의 「돈을 많이 가지고 있」까지 원문과 같고 그 뒤를 다르게 이었습니다



n이 크면 같은 문장이 반복되고 원문을 그대로 베끼는 문장이 나옵니다. 문맥이 길수록 원문에서 그 문맥이 나온 곳이 한두 군데뿐이라서 다음 토큰의 후보가 하나뿐인 경우가 많기 때문입니다.
데이터가 적을 때 생기는 과적합(overfitting)입니다. 문맥 길이마다 후보가 하나뿐인 문맥의 비율을 셉니다.

In [11]:
for n in range(2, 6):
    single = sum(len(c) == 1 for c in tables[n].values()) / len(tables[n])
    print(f"{n}-그램 (앞 토큰 {n - 1}개): 다음 후보가 하나뿐인 문맥 {single:.0%}")

2-그램 (앞 토큰 1개): 다음 후보가 하나뿐인 문맥 39%
3-그램 (앞 토큰 2개): 다음 후보가 하나뿐인 문맥 69%
4-그램 (앞 토큰 3개): 다음 후보가 하나뿐인 문맥 85%
5-그램 (앞 토큰 4개): 다음 후보가 하나뿐인 문맥 92%


## 백오프

후보가 하나뿐이면 원문을 베낄 수밖에 없습니다. 그럴 때는 문맥을 한 토큰 줄여서(n을 한 단계 낮춰서) 후보가 둘 이상인 표에서 뽑습니다. 이 전략이 백오프(backoff)입니다.

5-그램에서 시작해서 후보가 하나뿐이면 4-그램, 3-그램 순으로 내려갑니다. 2-그램까지 내려가면 후보가 하나여도 그대로 뽑습니다.
`trace`에 빈 리스트를 넘기면 토큰마다 어느 n-그램 표에서 뽑았는지 기록합니다.

In [12]:
def generate_backoff(tables, max_n, rng, prompt=(), max_len=100, trace=None):
    out = list(prompt)
    context = [BOS] * (max_n - 1) + out
    while len(out) < max_len:
        for n in range(max_n, 1, -1):    # max_n부터 2까지 내려가며
            counter = tables[n].get(tuple(context[-(n - 1):]))
            if counter and (len(counter) > 1 or n == 2):
                break                    # 후보가 둘 이상인 표를 찾으면 멈춤
        if not counter:
            break
        nxt = rng.choices(list(counter), weights=list(counter.values()))[0]
        if trace is not None:
            trace.append((n, len(counter), nxt))
        if nxt == EOS:
            break
        out.append(nxt)
        context.append(nxt)
    return out

rng = random.Random(1)
trace = []
ids = generate_backoff(tables, 5, rng, prompt, trace=trace)
print("만든 문장:", to_text(ids), "\n")
for n, k, nxt in trace:
    token = "<EOS>" if nxt == EOS else itos[nxt][0]
    print(f"{n}-그램 표 (후보 {k:>3}개) → {token}")

만든 문장: 돈을 모으면 보통 사람이 가지지 못하고 있다. 

5-그램 표 (후보   7개) → 모으
3-그램 표 (후보  10개) → 면
2-그램 표 (후보 770개) → 보통
2-그램 표 (후보  33개) → 사람
3-그램 표 (후보   7개) → 이
4-그램 표 (후보   5개) → 가지
4-그램 표 (후보   2개) → 지
5-그램 표 (후보   2개) → 못하
3-그램 표 (후보  47개) → 고
4-그램 표 (후보  23개) → 있
5-그램 표 (후보   7개) → 다
4-그램 표 (후보   8개) → .
2-그램 표 (후보   1개) → <EOS>


5-그램 표에서 뽑은 토큰과, 후보가 하나뿐이어서 더 낮은 표로 내려가 뽑은 토큰이 섞여 있습니다.
같은 방법으로 30문장을 만들어 앞의 5-그램 결과와 비교합니다.

In [13]:
rng = random.Random(1)
made_bo = [generate_backoff(tables, 5, rng, prompt) for _ in range(30)]
report(made_bo)

30문장 중 서로 다른 문장 30개 · 원문과 똑같은 문장 0개

   돈을 모으면 보통 사람이 가지지 못하고 있다.
   돈을 모아 말하고 발을 버둥거리는 거일세.
   돈을 줘 동굴 안으로 뛰어들어 쇼가 굉장한 속력을 낮추느냐는 등 헤아릴 때엔 가 닿기도하고, 주피터 호가 침몰한 곳에 닿게해 주는 일에 참가해 봤다.
   돈을 바꿔서 물 위를 기어가서 산 인간 사다리는 길게 끌어당기고 있는 것이 여러 가지로 결합을 파괴했으니까 찰리.
   돈을 주지 않으며 말했다.
   돈을 모아 셜록 홈즈의 모험이었던 거입니다.
   돈을 줘 때렸다.
   돈을 두 사람에게 간단히 자기 몸을 둥글게 구부렸기도합니다.
   돈을 줬고 혁대에 누군가가 살아 있다거나 했을 때의 자기 집에 들어가자 대통령을 상대로 해서 시간 탑이 윙윙거리며 돌아갔다.
   돈을 몽땅 따서 먹습니다.


백오프를 넣으면 같은 문장의 반복과 원문을 그대로 베낀 문장이 줄어듭니다. 대신 문맥을 줄인 만큼 문장이 덜 매끄러워지기도 합니다.

## 정리

- 공백과 문장부호도 글자로 세면 같은 코드로 문장을 만들 수 있습니다.
- 한국어는 형태소 단위로 나누면 어휘 수가 크게 줄어듭니다. 토큰은 숫자 ID로 다룹니다.
- n이 크면 원문을 베끼고(과적합), 백오프로 n을 낮추면 다양해지지만 문맥을 덜 봅니다.

n-그램은 표에 있는 문맥만 압니다. 본 적 없는 문맥 앞에서는 멈추거나, 백오프처럼 문맥의 일부를 버려야 합니다.
파트2에서는 뉴럴 네트워크로 이 문제를 다룹니다.

**더 해 볼 것:** 시작 단어(「돈을」)를 바꿔 보거나, 백오프의 `max_n`을 바꿔 보거나, `data/`에 다른 텍스트를 넣어 보세요.